<a href="https://colab.research.google.com/github/nobtax/mac237-labs/blob/main/lab2/MAC237_LAB2_S05_Intrusion_Detection_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MAC 237 Graded Lab 2, Session 5: reading an authentication log

**Wednesday, September 30, 2026. This session runs in Google Colab, in a browser.**

B124 is not available to us on Wednesday evenings, so this session is a notebook you can
run on any computer with a browser: a classroom machine or your own laptop. It uses only
the Python 3 standard library, needs no administrative rights, opens no socket, and reads
no file it did not write.

**Nothing in this notebook contacts any host outside it.** The Monday session built and
measured a service on the loopback address of a machine you controlled. This session does
the intrusion-detection half of the same story, but the events you read are ones this
notebook writes for you, so nothing has to be running and nothing has to be captured.

The point of the session is not the log format. It is the difference between counting
events and drawing a conclusion from them, which is the whole of intrusion detection.
The Kali version of this work, where you generate real `sshd` events on a machine you own
and read them from the journal, moves into Lab 2 in a room-based term; here you generate
the same shape of events in Python so the reasoning stays visible.

## Section 1. Your personalization token

Put your own student ID in `sid` and run the cell. The token prints into the output, and
the output is what gets saved, so it travels with the notebook. It must be visible in the
first and the last screenshot of this session.

In [1]:
import hashlib, datetime

sid = '24521159'          # put your own student ID here, in quotes
stamp = datetime.datetime.now().strftime('%Y%m%d-%H%M')
TOKEN = hashlib.sha256((sid + '-' + stamp).encode()).hexdigest()[:16]
print('MAC237 Lab 2 token:', TOKEN)

MAC237 Lab 2 token: 46ed8a8108bacedc


## Section 3. Part A. Generate the events

A real host records every authentication attempt whether it succeeded or failed. Here you
build a small log of that shape from your own token, so the events are yours and nobody
else's. The source addresses are all in the ranges RFC 5737 reserves for documentation, so
none of them is a real host, and the usernames are ordinary account names, not anyone's.

Run the cell. It writes twenty-two lines that look like the ones `journalctl -u ssh` would
print: a burst of failures from one address trying common usernames, a few scattered
failures, and exactly one accepted login, yours.

In [2]:
import random, datetime

seed = int(TOKEN, 16)
rng = random.Random(seed)

# Addresses only from the RFC 5737 documentation ranges, never a real host.
NOISY   = '198.51.100.' + str(10 + seed % 40)      # one address, many tries
SCATTER = ['203.0.113.' + str(5 + (seed >> b) % 30) for b in (1, 5, 9)]
USERS   = ['admin', 'root', 'test', 'oracle', 'postgres', 'git', 'ubuntu']
ME      = 'student' + str(seed % 100)

start = datetime.datetime(2026, 9, 30, 18, rng.randint(5, 20), 0)
events = []
t = start

def stamp(dt):
    return dt.strftime('%b %e %H:%M:%S').replace('  ', ' ')

# The burst: one address works through the common usernames, twice each.
for u in USERS[:6]:
    for _ in range(2):
        t += datetime.timedelta(seconds=rng.randint(1, 4))
        events.append(f'{stamp(t)} host sshd[{rng.randint(2000, 2999)}]: '
                      f'Failed password for invalid user {u} from {NOISY} port '
                      f'{rng.randint(40000, 60000)} ssh2')

# A little scatter from other addresses.
for addr in SCATTER:
    t += datetime.timedelta(seconds=rng.randint(5, 20))
    events.append(f'{stamp(t)} host sshd[{rng.randint(2000, 2999)}]: '
                  f'Failed password for {rng.choice(USERS)} from {addr} port '
                  f'{rng.randint(40000, 60000)} ssh2')

# Exactly one real login, yours, by password.
t += datetime.timedelta(seconds=rng.randint(3, 8))
events.append(f'{stamp(t)} host sshd[{rng.randint(2000, 2999)}]: '
              f'Accepted password for {ME} from 192.0.2.{seed % 50 + 1} port '
              f'{rng.randint(40000, 60000)} ssh2')

open('ssh_events.log', 'w').write('\n'.join(events) + '\n')
print('token:', TOKEN)
print('wrote', len(events), 'events to ssh_events.log')
print()
print('\n'.join(events[:4]))
print('...')
print(events[-1])

token: 46ed8a8108bacedc
wrote 16 events to ssh_events.log

Sep 30 18:06:01 host sshd[2208]: Failed password for invalid user admin from 198.51.100.14 port 43375 ssh2
Sep 30 18:06:05 host sshd[2826]: Failed password for invalid user admin from 198.51.100.14 port 54176 ssh2
Sep 30 18:06:08 host sshd[2786]: Failed password for invalid user root from 198.51.100.14 port 41510 ssh2
Sep 30 18:06:10 host sshd[2671]: Failed password for invalid user root from 198.51.100.14 port 43789 ssh2
...
Sep 30 18:07:19 host sshd[2293]: Accepted password for student44 from 192.0.2.45 port 56608 ssh2


## Section 3. Part B. Read three signals out of them

Now read the log the way an analyst does: not line by line, but as three questions asked of
the whole file. Which source address is responsible for the most failures? Which usernames
were tried? And when did the attempts cluster? The code below answers all three in the
Python standard library, so you will still understand it when you read it again in six
months. Run it, then read the three signals and the one accepted login it prints.

In [3]:
import re, collections

lines = open('ssh_events.log', encoding='utf-8').read().splitlines()
FAIL  = re.compile(r'Failed password for (?:invalid user )?(\S+) from (\S+)')
OK    = re.compile(r'Accepted (\S+) for (\S+) from (\S+)')
STAMP = re.compile(r'^(\w{3}\s+\d+\s+\d{2}:\d{2})')

by_source, by_user, per_minute = (collections.Counter() for _ in range(3))
for line in lines:
    m = FAIL.search(line)
    if m:
        by_user[m.group(1)] += 1
        by_source[m.group(2)] += 1
        s = STAMP.match(line)
        if s:
            per_minute[s.group(1)] += 1

accepted = [OK.search(l).groups() for l in lines if OK.search(l)]

print('token:', TOKEN)
print('signal 1, failures by source address :', by_source.most_common(3))
print('signal 2, usernames attempted        :', by_user.most_common(5))
print('signal 3, busiest minute             :', per_minute.most_common(1))
print('accepted logins (method, user, from) :', accepted)

top_source, top_count = by_source.most_common(1)[0]
assert top_count >= 8, 'the noisy address should dominate the failures'
assert len(accepted) == 1, 'there should be exactly one accepted login'
assert accepted[0][0] == 'password', 'the one login should be by password, not key'
print()
print('  CHECK PASS: one source dominates the failures')
print('  CHECK PASS: exactly one accepted login, yours, by password')
print('  CHECK PASS: the three signals were read from the whole file, not one line')

token: 46ed8a8108bacedc
signal 1, failures by source address : [('198.51.100.14', 12), ('203.0.113.17', 1), ('203.0.113.33', 1)]
signal 2, usernames attempted        : [('test', 3), ('git', 3), ('admin', 2), ('root', 2), ('oracle', 2)]
signal 3, busiest minute             : [('Sep 30 18:06', 13)]
accepted logins (method, user, from) : [('password', 'student44', '192.0.2.45')]

  CHECK PASS: one source dominates the failures
  CHECK PASS: exactly one accepted login, yours, by password
  CHECK PASS: the three signals were read from the whole file, not one line


## What goes in the report

Three `CHECK PASS` lines must appear above. If one does not, reread the message it printed;
the fix is in the cell it names, not in the assertion.

Answer these in the report:

1. Give the three signals your run produced. For each one, write one or two sentences
   saying what an attacker would have been doing and what further evidence would confirm
   or refute it. A count is not a finding.
2. Your own failures sit in the log next to everything else. On a production host with a
   thousand users, which of your three checks would still work, and which would drown in
   normal noise? Defend the answer.
3. Exactly one login was accepted, and it was by password rather than by key. Say in one
   sentence why an analyst treats an accepted password login after a burst of failures
   differently from an accepted key login.
4. Cite CTPE Chapter 8 on intrusion detection, and say in one sentence where a honeypot
   would sit in this picture and what it would add that this log does not.

**Screenshots**, both showing your Google account name in the browser and your token:
`lab2_s05_events.png`, the generated log with the failures and the one acceptance; and
`lab2_s05_signals.png`, the three signal lines and the `CHECK PASS` lines.

**Save and sign out.** Save the notebook to your repository with File, then Save a copy in
GitHub, into `lab2` under this notebook's name. Then choose Runtime, then Disconnect and
delete runtime, and on a shared computer sign out of Google in the browser. Capture
`lab2_s05_logout.png` while the token is still on screen. Upload the three screenshots to
`lab2` on github.com with Add file, then Upload files. If you worked on your own machine,
say so in the report; it does not affect your marks.